# Parameter Golf — Colab / Kaggle Launcher

**What this notebook does:** Clones the public parameter-golf repo, downloads data, and runs `train_gpt.py` via torchrun. GPU config is auto-selected by compute capability.

**Works on:** Google Colab and Kaggle Notebooks. Auto-detects environment and GPU.

> **Kaggle:** Enable *Settings → Internet* before running Cell 4.

## GPU tiers

| GPU | Compute cap | Script | Model | Expected BPB (10 min) |
|-----|-------------|--------|-------|-----------------------|
| T4 (free) | SM75 | root `train_gpt.py` | 6L 384d MHA | ~1.5–1.7 |
| P100 (Kaggle free) | SM60 | root `train_gpt.py` | 6L 384d MHA | ~1.4–1.6 |
| A100 / L4 / G4 | SM80–89 | root `train_gpt.py` | 9L 512d GQA | ~1.2–1.35 |
| H100 / Blackwell | SM90–120 | root `train_gpt.py` | 11L 512d GQA | ~1.15–1.30 |

## Known limitations on Colab/Kaggle
- Records scripts (GPTQ int6, sliding-window eval) require pushing local files to GitHub first
- torch.compile disabled on T4/P100 (no bfloat16) and Blackwell (Triton not yet tuned for SM120)
- Flash SDPA patched to math mode on pre-Ampere GPUs after clone


In [ ]:
# Cell 2: Check GPU
!nvidia-smi | head -12
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f"\nGPU: {gpu}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 3: Install dependencies
import torch

!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
print(f"GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}")
print("Dependencies installed. FA3 not needed — using root train_gpt.py (PyTorch SDPA).")


In [ ]:
# Cell 4: Detect environment, clone repo, patch for GPU
import os, subprocess, torch

# Auto-detect Colab vs Kaggle
if os.path.isdir('/content'):
    BASE = '/content'
    print('Environment: Google Colab')
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
    print('Environment: Kaggle')
    print('NOTE: Make sure Internet is ON in Settings → Internet')
else:
    BASE = os.path.expanduser('~')
    print(f'Environment: unknown, using {BASE}')

REPO   = f'{BASE}/parameter-golf'
FORK   = 'https://github.com/hwangmic-mueon/parameter-golf.git'
BRANCH = 'codex_loop_v1'

# Check if existing clone points to the right remote; reclone if not
if os.path.isdir(REPO):
    current_remote = subprocess.check_output(
        ['git', '-C', REPO, 'remote', 'get-url', 'origin'], text=True).strip()
    if 'hwangmic-mueon' not in current_remote:
        import shutil, os as _os
        _os.chdir('/')
        shutil.rmtree(REPO)
        print(f"Stale clone detected ({current_remote}) — recloning from fork")

if not os.path.isdir(REPO):
    !git clone {FORK} {REPO}

# Always fetch and switch to our working branch so fixes are picked up
!git -C {REPO} fetch origin {BRANCH}
!git -C {REPO} checkout -B {BRANCH} origin/{BRANCH}
!git -C {REPO} log --oneline -3
%cd {REPO}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
ampere_or_newer = cc[0] >= 8
print(f'GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}  flash_sdp_ok: {ampere_or_newer}')

script = f'{REPO}/train_gpt.py'
txt = open(script).read()
patched = False

# ── Patch 1: flash→math SDPA for T4/P100 (pre-Ampere, compute cap < 8) ──────
if not ampere_or_newer:
    txt = txt.replace('enable_flash_sdp(True)',  'enable_flash_sdp(False)')
    txt = txt.replace('enable_math_sdp(False)',  'enable_math_sdp(True)')
    print('Patched: flash SDPA → math SDPA (required for T4/P100)')
    patched = True

# ── Patch 2: DDP static_graph=True (needed for MTP auxiliary heads) ──────────
# Without this, DDP crashes on warmup step 1 when MTP params don't get a
# gradient (they're guarded by `if self.training`). static_graph=True skips
# the bucket-rebuild check and also reduces DDP overhead ~3x.
OLD_DDP = 'DDP(compiled_model, device_ids=[local_rank], broadcast_buffers=False)'
NEW_DDP = 'DDP(compiled_model, device_ids=[local_rank], broadcast_buffers=False, static_graph=True)'
if OLD_DDP in txt:
    txt = txt.replace(OLD_DDP, NEW_DDP)
    print('Patched: DDP static_graph=True (fixes MTP crash, reduces overhead)')
    patched = True
elif NEW_DDP in txt:
    print('DDP static_graph already present (no patch needed)')

if patched:
    open(script, 'w').write(txt)
if not patched and ampere_or_newer:
    print('No patches needed')


In [ ]:
# Cell 5: Download data (~800MB, 3-5 min)
# --train-shards 1 is fine for smoke tests. Use --train-shards 80 for real runs.
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1


In [ ]:
# Cell 6: Train
import subprocess, sys, os, torch, re

if os.path.isdir('/content'):
    BASE = '/content'
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
else:
    BASE = os.path.expanduser('~')

REPO = f'{BASE}/parameter-golf'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
is_hopper_plus    = cc[0] >= 9
is_ampere_plus    = cc[0] >= 8
is_blackwell_plus = cc[0] >= 12
print(f"GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}")

# Use ImprovedHyperparams records script on H100 (SM90 exact) if FA3 available
fa3_available = False
try:
    from flash_attn_interface import flash_attn_func
    fa3_available = True
except ImportError:
    pass

# FA3 wheels are built for H100 SM90 only — Blackwell SM120 has no kernel image at runtime
is_hopper_exact = cc[0] == 9
if is_hopper_exact and fa3_available:
    SCRIPT = f'{REPO}/records/track_10min_16mb/2026-03-26_ImprovedHyperparams/train_gpt.py'
    print("Using ImprovedHyperparams records script (FA3 + int6+lzma + n-gram + sliding-window)")
else:
    SCRIPT = f'{REPO}/train_gpt.py'
    if is_blackwell_plus and fa3_available:
        print("Using root train_gpt.py (FA3 installed but SM120 kernel unavailable)")
    print("Using root train_gpt.py (base script)")
print(f"Script: {SCRIPT}")

# Disable torch.compile on T4/P100 (no bfloat16) and Blackwell (Triton not tuned yet)
if not is_ampere_plus or is_blackwell_plus:
    os.environ['TORCHDYNAMO_DISABLE'] = '1'
    reason = 'Blackwell SM120+' if is_blackwell_plus else 'pre-Ampere'
    print(f"TORCHDYNAMO_DISABLE=1 ({reason})")

env = os.environ.copy()
env.update({
    'RUN_ID':                'colab_run',
    'MAX_WALLCLOCK_SECONDS': '600',
    'MUON_BACKEND_STEPS':    '10',
    'XSA_LAST_N':            '99',
    'BIGRAM_VOCAB_SIZE':     '2048',
    'SMEAR_ENABLED':         '1',
    'LN_SCALE':              '1',
    'ROPE_DIMS':             '16',
    'VE_ENABLED':            '1',   # vocabulary embedding injection (layer indices set per-tier)
    'MTP_NUM_HEADS':         '1',   # multi-token prediction auxiliary loss
    'VAL_LOSS_EVERY':        '500', # validate every 500 steps for better visibility
})

if is_blackwell_plus:
    # Blackwell: eager mode (Triton not tuned for SM120), int8+zlib compression
    # 7L 512d ≈ 14.8MB + VE + MTP ≈ 15.3MB — fits within 16MB budget.
    print("Model: 7L 512d GQA 8H/4KV")
    env.update({'NUM_LAYERS':'7','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'4',
                'TRAIN_BATCH_TOKENS':'131072','VAL_BATCH_SIZE':'131072','MLP_MULT':'3',
                'VE_LAYERS':'5,6,7'})
elif is_hopper_exact and fa3_available:
    # H100: records script handles model config — just set n-gram params.
    # N-gram defaults to off on single-GPU; force on since it's validated (alpha=0.40 order=7).
    env.update({'NGRAM_CACHE':'1','NGRAM_ALPHA':'0.40','NGRAM_ORDER':'7'})
elif is_hopper_plus:
    print("Model: 11L 512d GQA 8H/4KV")
    env.update({'NUM_LAYERS':'11','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'4',
                'TRAIN_BATCH_TOKENS':'786432','VAL_BATCH_SIZE':'524288','MLP_MULT':'3',
                'VE_LAYERS':'9,10,11'})
elif is_ampere_plus:
    print("Model: 9L 512d GQA 8H/4KV")
    env.update({'NUM_LAYERS':'9','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'4',
                'TRAIN_BATCH_TOKENS':'262144','VAL_BATCH_SIZE':'131072','MLP_MULT':'3',
                'VE_LAYERS':'7,8,9'})
else:
    print("Model: 6L 384d MHA")
    env.update({'NUM_LAYERS':'6','MODEL_DIM':'384','NUM_HEADS':'6','NUM_KV_HEADS':'6',
                'TRAIN_BATCH_TOKENS':'131072','VAL_BATCH_SIZE':'65536','MLP_MULT':'3',
                'VE_LAYERS':'4,5,6'})

LOG = f'{BASE}/train_log.txt'
cmd = ['torchrun','--standalone','--nproc_per_node=1', SCRIPT]
print(f"Log: {LOG}\n")

with open(LOG, 'w') as lf:
    proc = subprocess.Popen(cmd, env=env, cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush(); lf.write(line)
    proc.wait()
print(f"Exit code: {proc.returncode}")


In [ ]:
# Cell 7: Parse results from log
import re, os

if os.path.isdir('/content'):
    BASE = '/content'
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
else:
    BASE = os.path.expanduser('~')

LOG = f'{BASE}/train_log.txt'
try:
    txt = open(LOG).read()
except FileNotFoundError:
    txt = ''
    print('Log not found — did Cell 6 complete?')

print('=' * 55)
print('RESULTS')
print('=' * 55)

# Final BPB — look for the last occurrence
bpb_all = re.findall(r'val_bpb:([\d.]+)', txt)
print(f'Final val_bpb  : {bpb_all[-1] if bpb_all else "not found"}')

# Sliding window (SOTA script only)
sw = re.findall(r'final_int6_sliding_window_s64_exact.*val_bpb:([\d.]+)', txt)
if sw:
    print(f'Sliding window : {sw[-1]} (stride=64)')

# Artifact size
art = re.findall(r'Total submission size.*?([\d,]+) bytes', txt)
if art:
    b = int(art[-1].replace(',',''))
    print(f'Artifact size  : {b:,} bytes ({b/1e6:.2f} MB / 16.00 MB budget)')

# Key milestone activation
print()
for label, pat in [
    ('Late QAT activated',   r'late_qat:enabled step:(\d+)'),
    ('SWA started',          r'swa:start step:(\d+)'),
    ('muon_steps',           r'muon_steps:(\d+)'),
]:
    m = re.search(pat, txt)
    print(f'{label:20s}: {m.group(1) if m else "not found in log"}')

# Last 20 lines
print()
print('--- last 20 log lines ---')
for l in txt.strip().split('\n')[-20:]:
    print(l)


## Notes

### What this notebook actually does
This is a **launcher** — the ~1500-line training code lives in the `.py` files. The notebook configures hyperparameters as env vars and calls `torchrun` via subprocess, streaming output in real-time.

### Kaggle vs Colab
- Paths are auto-detected (`/content` for Colab, `/kaggle/working` for Kaggle)
- On Kaggle: enable **Settings → Internet** before running Cell 4
- Kaggle gives **30 GPU-hrs/week free** (T4 or P100)
- Colab Free gives T4 with session time limits (~1-2 hrs)

### Why T4/P100 scores are weak
T4 is ~10x slower than H100 for this workload. A 10-min T4 run processes ~600K tokens vs ~78M on 8×H100. The model barely starts to converge. Use T4/P100 to **verify the pipeline runs without errors**, not for competition scores.

### For a real leaderboard submission (8×H100 on RunPod)
1. Flash Attention 3 is **required** for the SOTA records script:
   ```bash
   pip install flash_attn_3 --find-links https://windreamer.github.io/flash-attention3-wheels/cu128_torch291
   ```
2. Use `--train-shards 80` when downloading data
3. Use `--nproc_per_node=8`
4. Our improved submission is at `records/track_10min_16mb/2026-03-26_ImprovedHyperparams/train_gpt.py`

### N-gram interpolation (validated on 8×H100)
The ImprovedHyperparams records script includes n-gram interpolation at eval time:
- Builds a multi-order backoff n-gram cache (orders 2–7) from training tokens
- Entropy-adaptive alpha: blends n-gram more heavily on predictable tokens
- **Validated**: alpha=0.40 order=7 → **1.0336 BPB** (vs 1.0517 without optimal params)
- Does **not** add to artifact size — cache is in-memory only, not serialized
- Auto-enabled on multi-GPU (8×H100); Cell 6 forces it on for single H100 too

### SOTA comparison
| Submission | BPB | Notes |
|-----------|-----|-------|
| SOTA (2026-03-24) | 1.1154 | 11L 512d, 8×H100, FA3, GPTQ int6 |
| Our pr638 base + n-gram | 1.0336 | Validated on 8×H100 with alpha=0.40 order=7 |
| Our target (ImprovedHyperparams + n-gram) | <1.03 | Must beat SOTA by 0.0072 to qualify |
| Colab Blackwell (7L, int8+zlib) | ~1.33–1.37 | Single GPU, no FA3, for pipeline testing only |
